# 2. Entrenamiento del modelo final
Entrena y guarda el modelo final.

**Modelo final** = promedio de 3 modelos de boosting + ponderación con un LLM en las publicaciones dudosas.

| Modelo | Variables | Semillas |
|---|---|---|
| LightGBM | 133 | 3 |
| XGBoost | 133 | 2 |
| CatBoost | 126 (`seller_id` y `category_id` como categóricas nativas) | 1 |

**Entradas** (en esta carpeta):
- `train_data.jsonlines`, `test_data.jsonlines` (datos de la competencia; pueden estar en la carpeta de arriba).
- `preds/oof_*` y `preds/test_*` de `bert`, `beto` y `xlmr`: salida de las notebooks `1a`, `1b` y `1c`.
- `llm_dudosas_scores_v2.csv`: puntajes del LLM para las publicaciones dudosas.

**Salidas** (en `modelos/`): los 6 archivos de modelos entrenados con todo train (una semilla cada uno: 3 LightGBM, 2 XGBoost, 1 CatBoost) y `ponderacion_llm.pkl`.

⏱ La primera vez calcula las variables de texto y de vecinos (~30 min) y las deja en `cache_*.pkl`; la notebook 3 las reutiliza. El entrenamiento tarda ~30 min (se usa 1 hilo para que el resultado sea idéntico entre corridas).

In [1]:
import os, pickle, time, warnings
import numpy as np, pandas as pd
from tp3_utils import *
from tp3_utils import _logit
warnings.filterwarnings('ignore')

def buscar(nombre):                      # los .jsonlines pueden estar en esta carpeta o en la de arriba
    for p in [nombre, os.path.join('..', nombre)]:
        if os.path.exists(p): return p
    raise FileNotFoundError(nombre)
TRAIN, TEST = buscar('train_data.jsonlines'), buscar('test_data.jsonlines')
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
os.makedirs('modelos', exist_ok=True)

### Variables
Todas las funciones están en `tp3_utils.py`. Las variables que usan el target (target encodings, modelos lineales de texto) se calculan **dentro de cada fold**: para las filas de validación sólo se usan etiquetas de las filas de entrenamiento de ese fold.

In [2]:
d = cargar_todo(TRAIN, TEST)             # variables de la publicación, del vendedor y de la categoría
raw = cargar_raw(TRAIN, TEST)            # datos crudos aplanados
y = d['y'].values                        # 1 = usado, 0 = nuevo

# XLM-R: su probabilidad de 'usado' entra como una variable más (en logit, igual que BERT y BETO)
oof_xlmr, test_xlmr = np.load('preds/oof_xlmr.npy'), np.load('preds/test_xlmr.npy')
def con_xlmr(F, X, Xt):
    F = [(a, b, Xa.assign(xlmr=_logit(oof_xlmr[a])), np.asarray(ya), Xb.assign(xlmr=_logit(oof_xlmr[b])), np.asarray(yb)) for a, b, Xa, ya, Xb, yb in F]
    return F, X.assign(xlmr=_logit(oof_xlmr)), Xt.assign(xlmr=_logit(test_xlmr))

# LightGBM / XGBoost: set completo con target encodings calculados dentro de cada fold
F_T, X_T, Xt_T = con_xlmr(*armar_final_tv(d, raw, te=True, raw_cats=False, base='v3'))
# CatBoost: recibe seller_id y category_id como categóricas nativas (sin target encodings a mano)
F_C, X_C, Xt_C = con_xlmr(*armar_final_tv(d, raw, te=False, raw_cats=True, base='v2'))
CATC = list(CATS) + ['cat_seller_id', 'cat_category_id']
def prep(X):
    X = X.copy()
    for c in CATC: X[c] = X[c].astype(str)
    return X
F_C = [(a, b, prep(Xa), ya, prep(Xb), yb) for a, b, Xa, ya, Xb, yb in F_C]; X_C, Xt_C = prep(X_C), prep(Xt_C)
print('LightGBM / XGBoost:', X_T.shape[1], 'variables | CatBoost:', X_C.shape[1], 'variables')

LightGBM / XGBoost: 133 variables | CatBoost: 126 variables


### Hiperparámetros
LightGBM y XGBoost usan los hiperparámetros que encontró Optuna (búsqueda bayesiana maximizando la accuracy en 5 folds) en la etapa de selección de modelos; CatBoost usa valores fijos, sin optimizar. La cantidad de árboles se eligió por validación cruzada (menor log-loss); el modelo final, que se entrena con todo train, usa 10% más de árboles porque tiene 25% más de datos.

In [3]:
N_JOBS = 1
SEMILLAS  = {'lgbm': [42, 7, 123], 'xgb': [42, 7], 'cat': [42]}
N_ARBOLES = {'lgbm': 1300, 'xgb': 500, 'cat': 1000}
P_LGBM = dict(learning_rate=0.0373, num_leaves=216, max_depth=4, min_child_samples=19, subsample=0.893, subsample_freq=1,
              colsample_bytree=0.381, reg_lambda=5.47, reg_alpha=0.488, min_split_gain=0.281, verbose=-1, n_jobs=N_JOBS)
P_XGB = dict(tree_method='hist', enable_categorical=True, eval_metric='logloss', learning_rate=0.0316, max_depth=8, min_child_weight=3.2,
             subsample=0.807, colsample_bytree=0.404, gamma=0.61, reg_lambda=0.59, reg_alpha=0.77, n_jobs=N_JOBS)
P_CAT = dict(loss_function='Logloss', learning_rate=0.06, depth=6, l2_leaf_reg=3, random_strength=1, bagging_temperature=1, border_count=254,
             cat_features=CATC, thread_count=N_JOBS, verbose=0, allow_writing_files=False)
CREAR = {'lgbm': lambda s, n: LGBMClassifier(**P_LGBM, random_state=s, n_estimators=n),
         'xgb':  lambda s, n: XGBClassifier(**P_XGB, random_state=s, n_estimators=n),
         'cat':  lambda s, n: CatBoostClassifier(**P_CAT, random_seed=s, iterations=n)}
GUARDAR = {'lgbm': lambda m, s: m.booster_.save_model(f'modelos/lgbm_s{s}.txt'),
           'xgb':  lambda m, s: m.save_model(f'modelos/xgb_s{s}.json'),
           'cat':  lambda m, s: m.save_model(f'modelos/cat_s{s}.cbm')}

### Entrenamiento
Para cada algoritmo y cada semilla:
1. **Validación cruzada (5 folds)** → predicciones *out-of-fold* de train. Hacen falta para ajustar la ponderación con el LLM.
2. **Modelo final con todo train** → se guarda en `modelos/`.

In [4]:
def entrenar(nombre, folds, X):
    n = N_ARBOLES[nombre]; oofs = []; t0 = time.time()
    for s in SEMILLAS[nombre]:
        oof = np.zeros(len(y))
        for a, b, Xa, ya, Xb, yb in folds:
            oof[b] = CREAR[nombre](s, n).fit(Xa, ya).predict_proba(Xb)[:, 1]
        oofs.append(oof)
        GUARDAR[nombre](CREAR[nombre](s, int(n * 1.1)).fit(X, y), s)
        print(f'{nombre} semilla {s}: accuracy CV {((oof >= .5) == y).mean():.4f} | promedio de semillas {((np.mean(oofs, 0) >= .5) == y).mean():.4f} ({(time.time() - t0) / 60:.0f} min)', flush=True)
    return np.mean(oofs, 0)

OOF = {'lgbm': entrenar('lgbm', F_T, X_T), 'xgb': entrenar('xgb', F_T, X_T), 'cat': entrenar('cat', F_C, X_C)}

lgbm semilla 42: accuracy CV 0.9460 | promedio de semillas 0.9460 (3 min)


lgbm semilla 7: accuracy CV 0.9458 | promedio de semillas 0.9459 (6 min)


lgbm semilla 123: accuracy CV 0.9459 | promedio de semillas 0.9461 (8 min)


xgb semilla 42: accuracy CV 0.9459 | promedio de semillas 0.9459 (1 min)


xgb semilla 7: accuracy CV 0.9464 | promedio de semillas 0.9462 (3 min)


cat semilla 42: accuracy CV 0.9454 | promedio de semillas 0.9454 (12 min)


### Ensamble de los 3 modelos (validación)
Promedio simple de las probabilidades.

In [5]:
oof_ens = np.mean(list(OOF.values()), 0)
for k, v in OOF.items(): print(f'{k:5s} accuracy CV {((v >= .5) == y).mean():.4f}')
print(f'\nENSAMBLE de 3: accuracy CV {((oof_ens >= .5) == y).mean():.4f}')

lgbm  accuracy CV 0.9461
xgb   accuracy CV 0.9462
cat   accuracy CV 0.9454

ENSAMBLE de 3: accuracy CV 0.9465


### Ponderación con el LLM en las publicaciones dudosas
Las publicaciones dudosas se eligieron con una versión anterior del modelo (sin XLM-R): las que tenían probabilidad entre 20% y 80%. En ellas un LLM leyó la ficha completa y dio su probabilidad de 'usado' (`p_sonnet`). En el núcleo más dudoso (35%–65%) hay además una segunda lectura de un modelo más fuerte (`p_fuerte`).

La probabilidad del ensamble y las del LLM se combinan con una **regresión logística** ajustada sobre las dudosas de train (5.522 filas), usando las predicciones *out-of-fold* del ensamble. Son dos regresiones: una para las filas con las dos lecturas y otra para las que sólo tienen `p_sonnet`.

In [6]:
# Los puntajes del LLM son probabilidades entre 0 y 1; se pasan a logit (recortando a [0.02, 0.98])
logit_llm = lambda q: np.log(np.clip(q, .02, .98) / (1 - np.clip(q, .02, .98)))
def entradas_llm(p_modelo, S, cols):
    return np.column_stack([_logit(p_modelo[S.fila.values])] + [logit_llm(S[c].values) for c in cols])

S = pd.read_csv('llm_dudosas_scores_v2.csv'); S = S[S.base == 'train']
nucleo = S.p_fuerte.notna()
PONDERACION = {}
for nombre, filas, cols in [('banda', S[~nucleo], ['p_sonnet']), ('nucleo', S[nucleo], ['p_sonnet', 'p_fuerte'])]:
    yy = y[filas.fila.values]
    PONDERACION[nombre] = (LogisticRegression().fit(entradas_llm(oof_ens, filas, cols), yy), cols)
    print(f'{nombre}: {len(filas)} filas | pesos [ensamble, {", ".join(cols)}] =', PONDERACION[nombre][0].coef_.round(3)[0],
          f'| accuracy del ensamble solo en esas filas: {((oof_ens[filas.fila.values] >= .5) == yy).mean():.3f}')
pickle.dump(PONDERACION, open('modelos/ponderacion_llm.pkl', 'wb'))
print('\nGuardado en modelos/:', sorted(os.listdir('modelos')))

banda: 4238 filas | pesos [ensamble, p_sonnet] = [0.841 0.401] | accuracy del ensamble solo en esas filas: 0.700
nucleo: 1284 filas | pesos [ensamble, p_sonnet, p_fuerte] = [ 0.507 -0.046  0.555] | accuracy del ensamble solo en esas filas: 0.563

Guardado en modelos/: ['cat_s42.cbm', 'lgbm_s123.txt', 'lgbm_s42.txt', 'lgbm_s7.txt', 'ponderacion_llm.pkl', 'xgb_s42.json', 'xgb_s7.json']
